# 05 - Research improvements: multi-scale scoring and edge-aware loss (Phase 4)

**Research question.** The Phase-2/3 baseline trains a from-scratch
convolutional autoencoder with plain MSE on normal images only and scores
anomalies with the single-scale, full-resolution error `S = mean_c |I - Î|`
(notebook 04). Two classical improvements claim to do better:

1. **Multi-scale scoring** - defects that survive at coarse resolution (regions,
   shape deviations) may contribute little to a single full-resolution map;
   pooling input and reconstruction to several scales and averaging the score
   maps gives coarse evidence a vote (`src/anomaly/multiscale.py`).
2. **Edge-aware loss** - MSE treats all intensity errors equally, while object
   contours and texture boundaries (spatial gradients) carry much of "normal
   appearance"; adding a gradient-matching term to the loss may force the model
   to respect structure whose absence makes defects stand out
   (`src/losses/edge_aware.py`).

**Hypotheses (stated before measuring, tested only by the tables in this
notebook):**

- H1: multi-scale scoring (B) improves *pixel-level* metrics over the
  baseline (A) on the same trained model.
- H2: edge-aware training (C) improves metrics over (A) under identical
  scoring.
- H3: the combination (D) is at least as good as the better of B and C.

**Controlled 2 x 2 design - four configurations, exactly two training runs:**

| Config | Training loss | Scoring | Question isolated |
|---|---|---|---|
| **A - baseline** | MSE | single-scale `S` | reference (Phase 2/3 behaviour) |
| **B - multi-scale** | MSE | multi-scale `S_ms` | scoring change only (same checkpoint as A) |
| **C - edge-aware** | MSE + λ-gradient | single-scale `S` | training change only |
| **D - both** | MSE + λ-gradient | multi-scale `S_ms` | interaction |

A/B share one checkpoint and C/D share the other, so every difference between
A and B (or C and D) is caused by the scoring function alone; every difference
between A and C (or B and D) is caused by the loss alone.

**Research rules for this phase**

- Both models train from **random initialisation on `train/good` only**,
  same seed, same optimizer, same epochs - only the criterion differs.
- Evaluation settings are **identical** across configurations: one protocol
  (`src/evaluation/protocol.py`), the same percentiles, the same metrics.
- Thresholds come from **held-out normal validation scores only** - never from
  test images or ground-truth masks. Hyperparameters (λ, scales, weights,
  percentiles) are fixed a priori below; nothing is tuned on test data.
- Ground-truth masks are loaded for evaluation (and for choosing *display*
  examples) only - never for scoring, thresholds or tuning.
- Results are printed by executing this notebook; **no metric value is written
  into the text in advance**. Claims about improvements are only allowed when
  the measured tables support them.
- Smoke run: `NB05_EPOCHS=2 NB05_IMAGE_SIZE=64` as environment variables
  executes everything quickly (~3 min on CPU); defaults are the full
  experiment (30 epochs, 256x256, ~6 min of training on a Colab T4).

## Section 1 - Setup

Same bootstrap as notebooks 03/04: locate the repository, import every `src`
module this notebook needs, print per-file SHA-256 checksums (staleness guard
for Colab uploads), fix the seed and pick the device. Phase 4 adds three
source files - the guard now covers them too.

In [ ]:
# Locate the repository root on this runtime (same probe pattern as notebooks 01/03/04).
from pathlib import Path

REPO_DIR_NAME = "edge-aware-anomaly-detection"

# Priority order:
#   1. cwd / its parent   - local kernel, or a kernel started inside notebooks/
#   2. /content/<repo>    - whole repo uploaded via right-click -> Upload to Colab
#   3. Google Drive copy  - only if Drive was mounted and holds a repo copy
#   4. /content           - fallback when only src/ was uploaded
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content") / REPO_DIR_NAME,
    Path("/content/drive/MyDrive") / REPO_DIR_NAME,
    Path("/content"),
]


def _is_repo_root(path: Path) -> bool:
    return (path / "src" / "config.py").is_file() and (path / "notebooks").is_dir()


print("Probe results:")
hits = []
for root in CANDIDATE_ROOTS:
    hit = _is_repo_root(root)
    print(f"  {'HIT ' if hit else 'miss'}  {root}")
    if hit:
        hits.append(root)

if not hits:
    raise FileNotFoundError(
        "Repository root not found on this runtime.\n"
        "Fix: in the VS Code Explorer, right-click the repository folder "
        f"('{REPO_DIR_NAME}') -> 'Upload to Colab', then re-run this cell."
    )

REPO_ROOT = hits[0]
if len(hits) > 1:
    print(f"\nMultiple hits; using the first: {REPO_ROOT}")

missing = [
    name
    for name in ("src", "configs", "notebooks", "requirements.txt", ".gitignore")
    if not (REPO_ROOT / name).exists()
]
if missing:
    raise FileNotFoundError(
        f"Found the repo under {REPO_ROOT}, but {missing} is missing there.\n"
        "Only part of the repository was uploaded. Upload the WHOLE folder "
        f"('{REPO_DIR_NAME}'): right-click it -> 'Upload to Colab'."
    )

print(f"Current working directory: {Path.cwd()}")
print(f"Repository root:            {REPO_ROOT}")

In [ ]:
# Import the modules Phase 4 needs and print checksums (staleness guard).
import datetime
import json
import sys
from pathlib import Path

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch

import numpy as np

import src
from src import config as project_config
from src.config import (
    file_sha256,
    get_device,
    get_paths,
    load_experiment_config,
    save_experiment_config,
    set_seed,
)
from src.anomaly import anomaly_score_map, image_score, multiscale_anomaly_score_map, reconstruct
from src.data import (
    NormalImageDataset,
    build_transform,
    create_dataloaders,
    find_dataset_root,
    scan_category,
)
from src.evaluation import evaluate_configuration, load_model_from_checkpoint
from src.losses import EdgeAwareReconstructionLoss, ReconstructionLoss, per_image_mse
from src.models import Autoencoder, count_trainable_parameters
from src.training import Trainer, build_optimizer, write_run_metadata

SRC_MODULES = [
    project_config.__file__,
    Path(src.__file__).parent / "data" / "dataset.py",
    Path(src.__file__).parent / "models" / "autoencoder.py",
    Path(src.__file__).parent / "losses" / "reconstruction.py",
    Path(src.__file__).parent / "losses" / "edge_aware.py",
    Path(src.__file__).parent / "training" / "trainer.py",
    Path(src.__file__).parent / "anomaly" / "scoring.py",
    Path(src.__file__).parent / "anomaly" / "heatmap.py",
    Path(src.__file__).parent / "anomaly" / "multiscale.py",
    Path(src.__file__).parent / "evaluation" / "metrics.py",
    Path(src.__file__).parent / "evaluation" / "protocol.py",
]

src_file = Path(src.__file__).resolve()
repo_resolved = REPO_ROOT.resolve()
assert src_file.is_relative_to(repo_resolved), (
    f"Imported 'src' from {src_file}, which is not under {repo_resolved}. "
    "A different 'src' package is shadowing the project module - restart the "
    "kernel and re-run from the top."
)

print(f"src package     : {src_file}")
print(f"src.__version__ : {src.__version__}")
print()

missing_src = [Path(m) for m in SRC_MODULES if not Path(m).is_file()]
if missing_src:
    listed = "\n".join(f"  - {p}" for p in missing_src)
    raise FileNotFoundError(
        "Source file(s) missing on this runtime:\n"
        f"{listed}\n"
        "This is a STALE UPLOAD: the repository copy on the runtime is older than "
        "your local one (Phase 4 added src/anomaly/multiscale.py, "
        "src/losses/edge_aware.py and src/evaluation/protocol.py). Fix: in VS Code, "
        "right-click the repository folder -> 'Upload to Colab', then re-run this cell."
    )

for module_path in SRC_MODULES:
    path = Path(module_path)
    mtime = datetime.datetime.fromtimestamp(path.stat().st_mtime)
    print(
        f"{path.relative_to(repo_resolved)}  sha256 {project_config.file_sha256(path)}  "
        f"mtime {mtime:%Y-%m-%d %H:%M}"
    )

print("\nStaleness guard: re-upload the repository folder after editing src/ locally.")
SRC_OK = True

In [ ]:
# Reproducibility seed + device selection.
SEED = project_config.SEED
set_seed(SEED)
paths = get_paths()
DEVICE = get_device()

print(f"seed            : {SEED} (python/numpy/torch/CUDA, deterministic cuDNN)")
print(f"device          : {DEVICE}")
if DEVICE.type == "cpu":
    print("note: CUDA not available - training will be slower but identical in result.")
print(f"outputs root    : {paths.outputs}")
print(f"configs         : {paths.configs}")

## Section 2 - Data: normal images for training, test split for evaluation only

Identical data rules as Phase 3:

- `train/good` produces the dataloaders (training + held-out normal
  validation via `create_dataloaders`); it is the **only** source of gradients
  and of threshold statistics.
- the full test split (normal + defective + masks) is enumerated with
  `scan_category` for evaluation; masks are opened later, for metrics only.

In [ ]:
# Resolve the dataset root and scan bottle (train/good only feeds training).
import os

try:
    DATASET_ROOT = find_dataset_root()
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"{exc}\n"
        "This notebook needs the extracted MVTec AD dataset. Fix: run "
        "notebooks/02_dataset_inspection.ipynb first, or set MVTEC_ROOT to the "
        "folder that contains 'bottle', 'cable', ... and re-run this cell."
    ) from None
print(f"dataset root  : {DATASET_ROOT}")

report = scan_category(DATASET_ROOT, "bottle", verify_images=True)
print(report.summary())
if report.errors:
    raise AssertionError("Dataset errors - fix before the experiment:\n" + "\n".join(report.errors))

print(f"train/good     : {len(report.train_good)}  <- ONLY training material")
print(f"test/good      : {len(report.test_good)}")
print(f"test defective : {report.n_test_defective}")
assert report.ok and len(report.train_good) > 0
SCAN_OK = True

In [ ]:
# Load the experiment config; optional environment overrides for smoke runs.
# NB05_EPOCHS / NB05_IMAGE_SIZE exist so CI-style local checks can run the
# whole notebook quickly; the default (no variables set) is the full experiment.
OVERRIDES = {}
if os.environ.get("NB05_EPOCHS"):
    OVERRIDES.setdefault("train", {})["epochs"] = int(os.environ["NB05_EPOCHS"])
    print(f"smoke override: train.epochs = {OVERRIDES['train']['epochs']}")
if os.environ.get("NB05_IMAGE_SIZE"):
    OVERRIDES.setdefault("data", {})["image_size"] = int(os.environ["NB05_IMAGE_SIZE"])
    print(f"smoke override: data.image_size = {OVERRIDES['data']['image_size']}")

cfg = load_experiment_config(paths.configs / "default.yaml", overrides=OVERRIDES or None)
print(f"experiment     : {cfg.experiment_name} (Phase-4 runs get their own names below)")
print(f"category       : {cfg.category}")
print(f"seed           : {cfg.seed}")
print(f"image_size     : {cfg.data.image_size}  (divisible by 2**{cfg.model.num_downsamples} = {cfg.model.downsample_factor})")
print(f"batch_size     : {cfg.data.batch_size}   val_fraction: {cfg.data.val_fraction}")
print(f"model          : base={cfg.model.base_channels} mults={list(cfg.model.channel_mults)} "
      f"latent={cfg.model.latent_channels} convs/stage={cfg.model.num_convs_per_stage}")
print(f"optimizer      : {cfg.train.optimizer}  lr={cfg.train.lr}  epochs={cfg.train.epochs}  grad_clip={cfg.train.grad_clip}")
CFG_OK = True

In [ ]:
# Train/validation dataloaders from train/good ONLY + split audit.
import sys as _sys

num_workers = cfg.data.num_workers
if _sys.platform.startswith("win") and num_workers > 0:
    print("Windows detected: forcing num_workers=0 (notebook + spawned workers clash).")
    num_workers = 0

train_loader, val_loader, split_info = create_dataloaders(
    DATASET_ROOT,
    category=cfg.category,
    image_size=cfg.data.image_size,
    batch_size=cfg.data.batch_size,
    val_fraction=cfg.data.val_fraction,
    seed=cfg.seed,
    num_workers=num_workers,
    pin_memory=DEVICE.type == "cuda",
)


def _is_train_good(path_str: str) -> str:
    return "/train/good/" in Path(path_str).as_posix()


assert split_info["n_train"] + split_info["n_val"] == split_info["n_all_train_good"]
assert all(_is_train_good(p) for p in split_info["train_paths"]), "train path outside train/good!"
assert all(_is_train_good(p) for p in split_info["val_paths"]), "val path outside train/good!"
assert val_loader is not None, "val_fraction=0 would remove the threshold source - keep >= 0.1"

VAL_PATHS = [Path(p) for p in split_info["val_paths"]]
print(f"train images   : {split_info['n_train']} ({len(train_loader)} batches/epoch)")
print(f"val images     : {split_info['n_val']} (threshold source + model selection)")
print("split audit    : every train/val path under train/good -> PASS")

transform = build_transform(cfg.data.image_size)
LOADERS_OK = True

## Section 3 - Improvement A: multi-scale anomaly scoring (`src/anomaly/multiscale.py`)

**Method.** For each integer downsample factor `s` in `scales = (1, 2, 4)`:

1. pool input and reconstruction to `(H/s, W/s)` (adaptive average pooling),
2. compute the baseline score there:
   `S_s(u, v) = (1/C) Σ_c |I_s(u, v, c) - Î_s(u, v, c)|`,
3. bilinearly upsample `S_s` to `(H, W)`.

Final score - a weighted average with `w` normalised to `Σ w_s = 1`:

```
S_ms(x, y) = Σ_s w_s · upsample(S_s)(x, y)
```

Default `weights = (0.5, 0.3, 0.2)`: the finest scale keeps the majority
vote (the baseline behaviour), coarser scales add region-level evidence.
Both `scales` and `weights` are **fixed a priori** (declared in the cell
below, recorded in `comparison.json`) and never optimised against test masks -
the same leakage rule as the Phase-3 threshold.

The function keeps the baseline contract (batched `(B, C, H, W)` in,
finite non-negative `(B, H, W)` out), so every downstream component -
thresholding, AUROC, Dice - works unchanged.

**What the verification cell checks:** `scales=(1,)` must reproduce the
baseline score *exactly*; a hand-computed 2-scale combination must match the
implementation; error paths must raise instead of returning garbage.

In [ ]:
# Step A verification: equivalence to baseline, hand-computed combination, error paths.
from src.anomaly import normalize_weights

results = []


def _raises(fn, exc):
    try:
        fn()
    except exc:
        return True
    except Exception as err:  # noqa: BLE001
        print(f"    wrong exception: {type(err).__name__}: {err}")
        return False
    return False


# build a deterministic toy pair
g_toy = torch.Generator().manual_seed(SEED)
toy_recon = torch.rand(2, 3, 16, 16, generator=g_toy)
toy_target = torch.rand(2, 3, 16, 16, generator=g_toy)

# 1) scale 1 only == baseline
only_s1 = multiscale_anomaly_score_map(toy_recon, toy_target, scales=(1,), weights=(1.0,))
baseline_s1 = anomaly_score_map(toy_recon, toy_target)
results.append(("scales=(1,) == baseline anomaly_score_map", torch.allclose(only_s1, baseline_s1, atol=1e-7)))
results.append(("output shape (2,16,16)", tuple(only_s1.shape) == (2, 16, 16)))
results.append(("finite + non-negative", bool(torch.isfinite(only_s1).all() and (only_s1 >= 0).all())))

# 2) hand-computed two-scale combination (pool manually with mean, not via the API)
hand_w = normalize_weights([3.0, 1.0])           # -> [0.75, 0.25]
s_fine = anomaly_score_map(toy_recon, toy_target)
pooled_recon = toy_recon.mean(dim=(2, 3), keepdim=True)   # global mean == 8x8? no: (2,3,1,1)
pooled_target = toy_target.mean(dim=(2, 3), keepdim=True)
s_coarse = anomaly_score_map(
    pooled_recon.expand(-1, -1, 16, 16) * torch.ones(1),
    pooled_target.expand(-1, -1, 16, 16) * torch.ones(1),
)  # constant images at full res: identical scores everywhere
hand = 0.75 * s_fine + 0.25 * s_coarse
impl = multiscale_anomaly_score_map(toy_recon, toy_target, scales=(1, 16), weights=(3.0, 1.0))
results.append(("hand-computed (0.75 fine + 0.25 coarse) matches", torch.allclose(impl, hand, atol=1e-6)))

# 3) weights normalise
results.append(("[2,2] weights -> [0.5,0.5]", abs(float(normalize_weights([2.0, 2.0]).sum()) - 1.0) < 1e-6))
results.append(("weight 0 raises", _raises(lambda: normalize_weights([1.0, 0.0]), ValueError)))
results.append(("scale 0 raises", _raises(lambda: multiscale_anomaly_score_map(toy_recon, toy_target, scales=(0,)), ValueError)))
results.append(("duplicate scales raise", _raises(lambda: multiscale_anomaly_score_map(toy_recon, toy_target, scales=(2, 2)), ValueError)))
results.append(("weights/scales length mismatch raises",
                _raises(lambda: multiscale_anomaly_score_map(toy_recon, toy_target, scales=(1, 2), weights=(1.0,)), ValueError)))
results.append(("shape mismatch raises",
                _raises(lambda: multiscale_anomaly_score_map(toy_recon, toy_target[:, :, :8, :8]), ValueError)))
results.append(("integer input raises TypeError",
                _raises(lambda: multiscale_anomaly_score_map(toy_recon.int(), toy_target), TypeError)))
results.append(("NaN reconstruction raises",
                _raises(lambda: multiscale_anomaly_score_map(
                    torch.full((1, 3, 8, 8), float("nan")), torch.zeros(1, 3, 8, 8)), ValueError)))

for name, ok in results:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in results), "multi-scale scoring verification failed"
print(f"\n{len(results)} checks passed.")
MULTISCALE_OK = True

## Section 4 - Improvement B: edge-aware reconstruction loss (`src/losses/edge_aware.py`)

**Method.** Let `G(x)` be the channel-averaged Sobel gradient magnitude
(3x3 kernels normalised by their L1 norm `8`, replicate padding, `sqrt(Gx²+Gy²)`
with an epsilon that keeps the backward pass finite on flat regions - one
scalar per pixel, shape `(B, H, W)`). The loss is

```
L = (1/N) Σ (x_i - x̂_i)²  +  λ · (1/N) Σ (G(x)_i - G(x̂)_i)²
      \___ pixel MSE ___/       \____ gradient matching ____/
```

- `λ = EDGE_WEIGHT`, fixed a priori (default **0.5**) - chosen before any
  evaluation, never tuned on test data. Because Sobel kernels are divided by
  8, `G` lives on a scale comparable to pixel values in `[0, 1]`, so `λ = O(1)`
  balances the two terms sensibly.
- `λ = 0` recovers plain MSE exactly (verified below), so the class is a
  strict generalisation of `ReconstructionLoss` and plugs into the **same**
  `Trainer` interface: `criterion(reconstruction, images) -> scalar`.
- Training targets: normal images only (Section 6/7), gradients never see
  defective images or masks.

**What the verification cell checks:** flat images have ~zero gradient
magnitude, a known ramp has the analytically expected response,
`edge_weight=0` equals `MSE` to numerical precision, the decomposition
`loss = MSE + λ·gradient-MSE` holds exactly, gradients flow, and bad inputs
raise.

In [ ]:
# Step B verification: gradient magnitude properties + loss algebra + error paths.
from src.losses import (
    EdgeAwareReconstructionLoss,
    gradient_magnitude_mse,
    spatial_gradient_magnitude,
)

results = []

# 1) flat inputs -> ~zero gradient (sqrt eps leaves only ~1e-6)
flat_zero = torch.zeros(1, 3, 8, 8)
flat_const = torch.full((1, 3, 8, 8), 0.7)
results.append(("gradient(zeros) < 1e-5", float(spatial_gradient_magnitude(flat_zero).max()) < 1e-5))
results.append(("gradient(constant) < 1e-5", float(spatial_gradient_magnitude(flat_const).max()) < 1e-5))

# 2) analytic ramp: x increases linearly along columns with step 1/7
#    Sobel(/8) responds with exactly the step size d on a linear ramp:
#    (r-l) weights give 8d/8 = d at every interior pixel.
ramp = torch.zeros(1, 1, 8, 8)
for j in range(8):
    ramp[0, 0, :, j] = j / 7.0
grad_center = float(spatial_gradient_magnitude(ramp)[0, 3, 3])
expected = 1.0 / 7.0
results.append((f"ramp gradient center ~ {expected:.4f} (got {grad_center:.4f})",
                abs(grad_center - expected) < 0.02))

# 3) loss algebra
g2 = torch.Generator().manual_seed(SEED + 1)
x_toy = torch.rand(2, 3, 32, 32, generator=g2)
xh_toy = torch.rand(2, 3, 32, 32, generator=g2)
mse_loss = ReconstructionLoss()(xh_toy, x_toy)
edge0 = EdgeAwareReconstructionLoss(edge_weight=0.0)(xh_toy, x_toy)
results.append(("edge_weight=0 == plain MSE", abs(float(edge0) - float(mse_loss)) < 1e-7))
edge_half = EdgeAwareReconstructionLoss(edge_weight=0.5)(xh_toy, x_toy)
g_term = gradient_magnitude_mse(xh_toy, x_toy)
results.append(("L == MSE + 0.5*gradMSE",
                abs(float(edge_half) - (float(mse_loss) + 0.5 * float(g_term))) < 1e-6))
results.append(("edge loss >= MSE (lambda > 0)", float(edge_half) >= float(mse_loss) - 1e-7))

# 4) gradients flow to the reconstruction
xh_grad = torch.rand(2, 3, 32, 32, generator=g2, requires_grad=True)
EdgeAwareReconstructionLoss(edge_weight=0.5)(xh_grad, x_toy).backward()
results.append(("backward finite grads", xh_grad.grad is not None and bool(torch.isfinite(xh_grad.grad).all())))

# 5) error paths
results.append(("shape mismatch raises", _raises(lambda: EdgeAwareReconstructionLoss(0.5)(xh_toy[:, :, :8], x_toy), ValueError)))
results.append(("negative lambda raises", _raises(lambda: EdgeAwareReconstructionLoss(edge_weight=-0.1), ValueError)))
results.append(("NaN input raises",
                _raises(lambda: EdgeAwareReconstructionLoss(0.5)(
                    torch.full((1, 3, 4, 4), float("nan")), torch.zeros(1, 3, 4, 4)), ValueError)))
results.append(("integer input raises TypeError",
                _raises(lambda: EdgeAwareReconstructionLoss(0.5)(
                    torch.zeros(1, 3, 4, 4, dtype=torch.int64), torch.zeros(1, 3, 4, 4)), TypeError)))
results.append(("non-float lambda raises TypeError",
                _raises(lambda: EdgeAwareReconstructionLoss(edge_weight="high"), TypeError)))

for name, ok in results:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in results), "edge-aware loss verification failed"
print(f"\n{len(results)} checks passed. criterion = {EdgeAwareReconstructionLoss(edge_weight=0.5)}")
EDGE_LOSS_OK = True

## Section 5 - Experiment matrix and shared settings

Everything that must stay constant across the four configurations is fixed
here, in one cell, so the comparison is auditable:

| Fixed a priori | Value | Why this value |
|---|---|---|
| seed | `cfg.seed` (42) | same data order *and* same initial weights for both training runs |
| optimizer / lr / epochs / batch / image size | `configs/default.yaml` | the Phase-2 baseline hyper-parameters - improvements must beat *this*, not a retuned variant |
| `EDGE_WEIGHT` (λ) | 0.5 | balances MSE and gradient-MSE on `[0, 1]` pixels (Sobel /8); not swept |
| `MS_SCALES` | (1, 2, 4) | full, 1/2 and 1/4 resolution at 256x256 |
| `MS_WEIGHTS` | (0.5, 0.3, 0.2) | finest scale keeps the majority vote |
| pixel / image percentiles | 99.5 / 95.0 | the Phase-3 threshold protocol, reused unchanged |
| metrics | image AUROC, pixel AUROC, pixel AP, Dice/IoU | `src/evaluation/metrics.py` (sklearn-validated in notebook 04) |

The **only** differences between configurations are the criterion used for
training (A/B: MSE; C/D: MSE + λ-gradient) and the scoring function at
evaluation (A/C: single-scale; B/D: multi-scale).

`PHASE4_OUTPUTS` optionally redirects all artifacts (used by the local
verification harness so synthetic runs never pollute the real `outputs/`
directory; leave it unset on Colab).

In [ ]:
# The 2x2 matrix + fixed constants + artifact directories.
OUTPUTS_ROOT = Path(os.environ["PHASE4_OUTPUTS"]) if os.environ.get("PHASE4_OUTPUTS") else paths.outputs

EDGE_WEIGHT = 0.5
MS_SCALES = (1, 2, 4)
MS_WEIGHTS = (0.5, 0.3, 0.2)
PIXEL_PERCENTILE = 99.5
IMAGE_PERCENTILE = 95.0


def score_single_scale(reconstruction, images):
    return anomaly_score_map(reconstruction, images)


def score_multi_scale(reconstruction, images):
    return multiscale_anomaly_score_map(reconstruction, images, scales=MS_SCALES, weights=MS_WEIGHTS)


EXPERIMENTS = [
    {"key": "A_baseline", "title": "A: baseline (MSE + single-scale)", "model": "baseline", "scoring": "single-scale", "score_fn": score_single_scale},
    {"key": "B_multiscale", "title": "B: multi-scale scoring", "model": "baseline", "scoring": "multi-scale", "score_fn": score_multi_scale},
    {"key": "C_edge", "title": "C: edge-aware loss", "model": "edge", "scoring": "single-scale", "score_fn": score_single_scale},
    {"key": "D_both", "title": "D: edge-aware + multi-scale", "model": "edge", "scoring": "multi-scale", "score_fn": score_multi_scale},
]

BASELINE_RUN_DIR = OUTPUTS_ROOT / "runs" / "research_baseline"
EDGE_RUN_DIR = OUTPUTS_ROOT / "runs" / "research_edge_aware"
PHASE4_DIR = OUTPUTS_ROOT / "runs" / "phase4_comparison"
FIGURES_DIR = PHASE4_DIR / "figures"
for directory in (BASELINE_RUN_DIR, EDGE_RUN_DIR, PHASE4_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"{'key':<14} {'training loss':<22} {'scoring':<12} checkpoint")
print("-" * 74)
print(f"{'A_baseline':<14} {'MSE':<22} {'single-scale':<12} research_baseline/checkpoints/best.pt")
print(f"{'B_multiscale':<14} {'MSE':<22} {'multi-scale':<12} (same checkpoint as A)")
print(f"{'C_edge':<14} {'MSE + 0.5*gradMSE':<22} {'single-scale':<12} research_edge_aware/checkpoints/best.pt")
print(f"{'D_both':<14} {'MSE + 0.5*gradMSE':<22} {'multi-scale':<12} (same checkpoint as C)")
print(f"\nlambda = {EDGE_WEIGHT}   scales = {MS_SCALES}   weights = {MS_WEIGHTS}")
print(f"thresholds: pixel P{PIXEL_PERCENTILE}, image P{IMAGE_PERCENTILE} of NORMAL val scores")
print(f"artifacts  : {PHASE4_DIR}")
MATRIX_OK = True

## Section 6 - Training run 1: the baseline (MSE), from random initialisation

`set_seed` is called immediately before the model is built, so the baseline
and the edge-aware run in Section 7 start from **identical random weights**
and see the **same batch order** (the DataLoader generator is seeded with the
same config seed inside `create_dataloaders`). Both runs therefore differ only
in their criterion - the controlled comparison.

Run directory: `outputs/runs/research_baseline/` (same artifact format as
Phase 2: `checkpoints/best.pt`, `history.json`, `config.yaml`).

In [ ]:
# Train the baseline: fresh model (seeded), MSE criterion, Trainer as in Phase 2.
import time

set_seed(cfg.seed)  # identical initial weights for both training runs

baseline_model = Autoencoder.from_config(cfg.model)
n_params = count_trainable_parameters(baseline_model)
print(f"baseline model parameters : {n_params:,}")

baseline_criterion = ReconstructionLoss(reduction="mean")
baseline_optimizer = build_optimizer(
    baseline_model, cfg.train.optimizer, cfg.train.lr, cfg.train.weight_decay
)

baseline_run_config = {
    **cfg.to_dict(),
    "experiment_name": "research_baseline",
}
save_experiment_config(
    load_experiment_config(overrides=baseline_run_config), BASELINE_RUN_DIR / "config.yaml"
)
(BASELINE_RUN_DIR / "split_info.json").write_text(
    json.dumps(split_info, indent=2), encoding="utf-8"
)

baseline_trainer = Trainer(
    baseline_model,
    baseline_criterion,
    baseline_optimizer,
    DEVICE,
    run_dir=BASELINE_RUN_DIR,
    grad_clip=cfg.train.grad_clip,
    config=baseline_run_config,
)
baseline_result = baseline_trainer.fit(train_loader, val_loader, epochs=cfg.train.epochs)

write_run_metadata(
    BASELINE_RUN_DIR,
    experiment_name="research_baseline",
    phase="4",
    training_loss="mse",
    edge_weight=0.0,
    seed=cfg.seed,
    dataset_root=str(DATASET_ROOT),
    image_size=cfg.data.image_size,
    epochs=cfg.train.epochs,
    best_metric=baseline_result.best_metric,
    training_seconds=baseline_result.total_seconds,
    device=str(DEVICE),
    trainable_parameters=n_params,
)

assert len(baseline_result.records) == cfg.train.epochs
assert all(r.train_loss == r.train_loss for r in baseline_result.records), "non-finite train loss"
assert (BASELINE_RUN_DIR / "checkpoints" / "best.pt").is_file()
print(f"best baseline val loss : {baseline_result.best_metric:.6f}")
print(f"training seconds       : {baseline_result.total_seconds:.1f}")
BASELINE_TRAINED = True

## Section 7 - Training run 2: the edge-aware autoencoder

Identical to Section 6 in every respect - same seed (so the same initial
weights), same dataloaders, optimizer, epochs - except the criterion:

```
EdgeAwareReconstructionLoss(edge_weight=0.5):  L = MSE + 0.5 * gradient_MSE
```

Run directory: `outputs/runs/research_edge_aware/`.

**Reading the training curves honestly:** the edge run's *recorded* loss
contains the extra gradient term, so its curve is **not** numerically
comparable to the baseline's MSE curve. After both runs, a common-currency
cell evaluates **plain MSE on the held-out normal validation set for both
models** - that is the fair training-quality comparison.

In [ ]:
# Train the edge-aware model: same seed -> same init; only the criterion differs.
set_seed(cfg.seed)

edge_model = Autoencoder.from_config(cfg.model)
assert count_trainable_parameters(edge_model) == n_params, "parameter count must match the baseline"

edge_criterion = EdgeAwareReconstructionLoss(edge_weight=EDGE_WEIGHT)
edge_optimizer = build_optimizer(
    edge_model, cfg.train.optimizer, cfg.train.lr, cfg.train.weight_decay
)
print(f"criterion : {edge_criterion}")

edge_run_config = {
    **cfg.to_dict(),
    "experiment_name": "research_edge_aware",
}
save_experiment_config(
    load_experiment_config(overrides=edge_run_config), EDGE_RUN_DIR / "config.yaml"
)
(EDGE_RUN_DIR / "split_info.json").write_text(json.dumps(split_info, indent=2), encoding="utf-8")

edge_trainer = Trainer(
    edge_model,
    edge_criterion,
    edge_optimizer,
    DEVICE,
    run_dir=EDGE_RUN_DIR,
    grad_clip=cfg.train.grad_clip,
    config=edge_run_config,
)
edge_result = edge_trainer.fit(train_loader, val_loader, epochs=cfg.train.epochs)

write_run_metadata(
    EDGE_RUN_DIR,
    experiment_name="research_edge_aware",
    phase="4",
    training_loss="mse + edge_weight * gradient_magnitude_mse",
    edge_weight=EDGE_WEIGHT,
    seed=cfg.seed,
    dataset_root=str(DATASET_ROOT),
    image_size=cfg.data.image_size,
    epochs=cfg.train.epochs,
    best_metric=edge_result.best_metric,
    training_seconds=edge_result.total_seconds,
    device=str(DEVICE),
    trainable_parameters=n_params,
)

assert len(edge_result.records) == cfg.train.epochs
assert all(r.train_loss == r.train_loss for r in edge_result.records), "non-finite train loss"
assert (EDGE_RUN_DIR / "checkpoints" / "best.pt").is_file()
print(f"best edge run recorded loss (MSE+grad term): {edge_result.best_metric:.6f}")
print(f"training seconds                            : {edge_result.total_seconds:.1f}")
EDGE_TRAINED = True

In [ ]:
# Training curves (recorded losses, labelled honestly) + common-currency val MSE.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, result, title in (
    (axes[0], baseline_result, "baseline (criterion = MSE)"),
    (axes[1], edge_result, "edge-aware (criterion = MSE + 0.5*gradMSE)"),
):
    epochs = [r.epoch for r in result.records]
    axis.plot(epochs, [r.train_loss for r in result.records], label="train")
    if result.records[0].val_loss is not None:
        axis.plot(epochs, [r.val_loss for r in result.records], label="val")
    axis.set_xlabel("epoch")
    axis.set_ylabel("recorded loss")
    axis.set_title(title)
    axis.legend()
    axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150)
plt.show()
print(f"saved: {FIGURES_DIR / 'training_curves.png'}")


@torch.no_grad()
def plain_val_mse(model: torch.nn.Module) -> float:
    # Common-currency validation MSE for any model, no criterion involved.
    was_training = model.training
    model.eval()
    total, n = 0.0, 0
    for batch in val_loader:
        x = batch["image"].to(DEVICE)
        recon = model(x)
        total += float(per_image_mse(recon, x).sum())
        n += x.shape[0]
    model.train(was_training)
    return total / n


val_mse_baseline = plain_val_mse(baseline_trainer.model)
val_mse_edge = plain_val_mse(edge_trainer.model)
print(f"\nplain val MSE, baseline checkpoint : {val_mse_baseline:.6f}")
print(f"plain val MSE, edge-aware checkpoint: {val_mse_edge:.6f}")
print("(lower is better; identical metric for both models - unlike the recorded curves)")
COMMON_VAL_MSE_OK = True

## Section 8 - Shared evaluation protocol (`src/evaluation/protocol.py`)

Every configuration is evaluated by **one function**, so no configuration can
benefit from a kinder protocol:

1. **thresholds** - pixel P99.5 and image P95 of the score distribution of the
   held-out normal validation split only (the same `val_paths` used for model
   selection; all under `train/good`, asserted inside the function);
2. **test inference** - batched, label-free (`reconstruct` forces eval mode);
3. **ground truth** - masks nearest-resized to the score-map grid, evaluation
   only;
4. **metrics** - image AUROC, pooled pixel AUROC, pooled pixel AP, per-image
   Dice/IoU at the pixel threshold (defective-mean headline, empty-mask
   convention 1.0) - the notebook-04 definitions, unchanged.

For multi-scale configurations the *same* scoring function is used for the
threshold source and the test split: the percentile protocol is applied to
`S_ms` scores, so thresholds and test scores always speak the same language.

Checkpoints are reloaded from disk (source of truth) rather than reused from
the training cells, and only one trained model is held in memory at a time.

In [ ]:
# Build the test manifest (images + mask paths; masks not opened yet).
test_manifest = []
for path in report.test_good:
    test_manifest.append(
        {"path": path, "name": path.name, "defect": "good", "is_defect": False, "mask_path": None}
    )
for defect in sorted(report.test_by_defect):
    for path in report.test_by_defect[defect]:
        mask_path = DATASET_ROOT / cfg.category / "ground_truth" / defect / f"{path.stem}_mask.png"
        test_manifest.append(
            {"path": path, "name": path.name, "defect": defect, "is_defect": True, "mask_path": mask_path}
        )

missing_masks = [
    e["name"] for e in test_manifest if e["is_defect"] and not Path(e["mask_path"]).is_file()
]
assert not missing_masks, f"missing masks: {missing_masks}"
n_normal_test = sum(1 for e in test_manifest if not e["is_defect"])
n_defective_test = len(test_manifest) - n_normal_test
assert n_normal_test > 0 and n_defective_test > 0
print(f"test manifest   : {len(test_manifest)} images "
      f"({n_normal_test} normal, {n_defective_test} defective), masks verified present")
MANIFEST_OK = True

In [ ]:
# Evaluate all four configurations with the identical protocol.
BASELINE_CKPT = BASELINE_RUN_DIR / "checkpoints" / "best.pt"
EDGE_CKPT = EDGE_RUN_DIR / "checkpoints" / "best.pt"
assert BASELINE_CKPT.is_file() and EDGE_CKPT.is_file(), "training did not produce checkpoints"

RESULTS: dict = {}
CHECKPOINT_IDS: dict = {}
CHECKPOINT_PATHS = {"baseline": BASELINE_CKPT, "edge": EDGE_CKPT}

for model_key, ckpt_path in CHECKPOINT_PATHS.items():
    print(f"\n=== evaluating model '{model_key}' ({ckpt_path.name} @ {ckpt_path.parent.parent.name}) ===")
    model_loaded, payload = load_model_from_checkpoint(ckpt_path, Autoencoder, DEVICE)
    CHECKPOINT_IDS[model_key] = file_sha256(ckpt_path, length=12)
    for experiment in EXPERIMENTS:
        if experiment["model"] != model_key:
            continue
        result = evaluate_configuration(
            model_loaded,
            test_manifest,
            VAL_PATHS,
            transform,
            DEVICE,
            experiment["score_fn"],
            pixel_percentile=PIXEL_PERCENTILE,
            image_percentile=IMAGE_PERCENTILE,
            batch_size=cfg.data.batch_size,
        )
        RESULTS[experiment["key"]] = result
        m = result["metrics"]
        print(
            f"  {experiment['key']:<13} image AUROC {m['image_auroc']:.4f}  "
            f"pixel AUROC {m['pixel_auroc']:.4f}  AP {m['pixel_ap']:.4f}  "
            f"Dice {m['dice_defective_mean']:.4f}  IoU {m['iou_defective_mean']:.4f}  "
            f"(t_px={result['pixel_threshold']:.5f} from {result['n_val_source']} val images)"
        )
    del model_loaded
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

assert set(RESULTS) == {e["key"] for e in EXPERIMENTS}, "not every configuration was evaluated"
EVAL_OK = True
print("\nAll four configurations evaluated with the same protocol, thresholds and metrics.")

## Section 9 - Comparison table and automatic verdicts

The table below is the **only** evidence for or against H1/H2/H3. Deltas are
computed automatically against configuration A; nothing is rounded in favour
of any configuration. A single seed on a single category is enough to spot
*directional* evidence, not enough for a publication-grade claim - see the
limitations recorded with the results.

In [ ]:
# Side-by-side metric table + automatic delta-vs-baseline verdicts.
METRIC_COLUMNS = [
    ("image_auroc", "image AUROC", 4),
    ("pixel_auroc", "pixel AUROC", 4),
    ("pixel_ap", "pixel AP", 4),
    ("dice_defective_mean", "Dice (def)", 4),
    ("iou_defective_mean", "IoU (def)", 4),
    ("fp_normal_images", "FP normal imgs", 0),
]

header = f"{'config':<14}" + "".join(f" {label:>16}" for _, label, _ in METRIC_COLUMNS)
print(header)
print("-" * len(header))
baseline_metrics = RESULTS["A_baseline"]["metrics"]
for experiment in EXPERIMENTS:
    metrics = RESULTS[experiment["key"]]["metrics"]
    row = f"{experiment['key']:<14}"
    for key, _, digits in METRIC_COLUMNS:
        value = metrics[key]
        row += f" {value:>16.{digits}f}" if digits else f" {int(value):>16d}"
    print(row)

print("\nDeltas vs A_baseline (positive = higher value; for FP columns lower is better):")
for experiment in EXPERIMENTS[1:]:
    metrics = RESULTS[experiment["key"]]["metrics"]
    print(f"\n  {experiment['key']}:")
    for key, label, _ in METRIC_COLUMNS:
        delta = float(metrics[key]) - float(baseline_metrics[key])
        marker = "  " if abs(delta) < 1e-12 else ("up" if delta > 0 else "dn")
        print(f"    {label:<18} {marker} {delta:+.4f}")

# Hypothesis verdicts (measured; eps guards float-noise-level differences)
_EPS = 1e-9


def _verdict(delta: float) -> str:
    if delta > _EPS:
        return "SUPPORTED"
    if delta < -_EPS:
        return "NOT supported"
    return "no measurable difference"


h1_pixel = RESULTS["B_multiscale"]["metrics"]["pixel_auroc"] - baseline_metrics["pixel_auroc"]
h1_image = RESULTS["B_multiscale"]["metrics"]["image_auroc"] - baseline_metrics["image_auroc"]
h2_pixel = RESULTS["C_edge"]["metrics"]["pixel_auroc"] - baseline_metrics["pixel_auroc"]
h2_dice = RESULTS["C_edge"]["metrics"]["dice_defective_mean"] - baseline_metrics["dice_defective_mean"]
best_single = max(
    RESULTS["B_multiscale"]["metrics"]["pixel_auroc"],
    RESULTS["C_edge"]["metrics"]["pixel_auroc"],
)
h3_pixel = RESULTS["D_both"]["metrics"]["pixel_auroc"] - best_single

print("\nHypothesis verdicts (from THIS run only):")
print(f"  H1 multi-scale improves pixel AUROC vs A : {_verdict(h1_pixel)} (delta {h1_pixel:+.4f})")
print(f"     ... and image AUROC                   : {_verdict(h1_image)} (delta {h1_image:+.4f})")
print(f"  H2 edge-aware improves pixel AUROC vs A  : {_verdict(h2_pixel)} (delta {h2_pixel:+.4f})")
print(f"     ... and Dice (defective mean)         : {_verdict(h2_dice)} (delta {h2_dice:+.4f})")
print(f"  H3 combined >= better single change (px) : {_verdict(h3_pixel)} (delta {h3_pixel:+.4f})")
print("\nThese verdicts describe one seed, one category, one training pair.")
print("Treat them as measured evidence, not as a general claim about the methods.")
TABLE_OK = True

In [ ]:
# Bar chart of the headline metrics per configuration (same axes, honest scale 0-1 where applicable).
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
bar_metrics = [
    ("image_auroc", "image AUROC"),
    ("pixel_auroc", "pixel AUROC"),
    ("pixel_ap", "pixel AP"),
    ("dice_defective_mean", "Dice (defective mean)"),
]
keys = [e["key"] for e in EXPERIMENTS]
short = [k.split("_", 1)[0] for k in keys]
for axis, (metric_key, title) in zip(axes, bar_metrics):
    values = [RESULTS[k]["metrics"][metric_key] for k in keys]
    bars = axis.bar(short, values, color=["#4c72b0", "#55a868", "#c44e52", "#8172b2"])
    axis.set_ylim(0.0, 1.0)
    axis.set_title(title, fontsize=10)
    axis.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f"{value:.3f}",
                  ha="center", fontsize=8)
fig.suptitle("Phase 4 - four configurations, identical evaluation protocol", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "metrics_bars.png", dpi=150)
plt.show()
print(f"saved: {FIGURES_DIR / 'metrics_bars.png'}")
BARS_OK = True

## Section 10 - Visual comparison on the same images

One figure per selected image: the input and its ground-truth mask in the
first two columns, then (reconstruction | heatmap | predicted mask) for each
configuration. Heatmaps are pinned to one `vmax` **per image across all four
configurations**, so colour intensity is comparable between rows; the
predicted masks use each configuration's own normal-validation threshold
(printed with the figure).

Image selection (display only - metrics never depend on it): the two
defective test images with the largest ground-truth area (one per defect type
when available) plus the normal image with the highest baseline image score
(the hardest normal case for configuration A). Selection uses ground truth
*for display curation only*, exactly like notebook 04's failure-case grids.

In [ ]:
# Render per-image comparison grids for the selected examples.
from src.anomaly import apply_threshold, score_to_heatmap

labels_np_all = np.array([e["is_defect"] for e in test_manifest], dtype=int)
defective_entries = [i for i, e in enumerate(test_manifest) if e["is_defect"]]
normal_entries = [i for i, e in enumerate(test_manifest) if not e["is_defect"]]

gt_areas = {i: int(RESULTS["A_baseline"]["gt_masks"][i].sum()) for i in defective_entries}
seen_types = set()
selected = []
for i in sorted(defective_entries, key=lambda i: gt_areas[i], reverse=True):
    defect_type = test_manifest[i]["defect"]
    if defect_type in seen_types:
        continue
    seen_types.add(defect_type)
    selected.append(i)
    if len(selected) == 2:
        break
worst_normal = max(normal_entries, key=lambda i: float(RESULTS["A_baseline"]["image_scores"][i]))
selected.append(worst_normal)
print("selected display images:",
      [(test_manifest[i]["defect"], test_manifest[i]["name"]) for i in selected])

sel_paths = [test_manifest[i]["path"] for i in selected]
sel_dataset = NormalImageDataset(sel_paths, transform)
sel_images = torch.stack([sel_dataset[k]["image"] for k in range(len(sel_dataset))]).to(DEVICE)

MODEL_FOR_VIZ = {"baseline": None, "edge": None}
for model_key, ckpt_path in CHECKPOINT_PATHS.items():
    MODEL_FOR_VIZ[model_key], _ = load_model_from_checkpoint(ckpt_path, Autoencoder, DEVICE)

with torch.no_grad():
    RECONS = {}
    SCORES = {}
    for model_key, model_viz in MODEL_FOR_VIZ.items():
        recon_batch = reconstruct(model_viz, sel_images)
        RECONS[model_key] = recon_batch.cpu()
        SCORES[model_key] = {
            "single-scale": anomaly_score_map(recon_batch, sel_images).cpu(),
            "multi-scale": multiscale_anomaly_score_map(
                recon_batch, sel_images, scales=MS_SCALES, weights=MS_WEIGHTS
            ).cpu(),
        }
    del MODEL_FOR_VIZ
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

heat_vmax = max(0.05, float(max(SCORES[m][s].max() for m in SCORES for s in SCORES[m])))

panel_titles = ["original", "ground truth"]
for experiment in EXPERIMENTS:
    panel_titles += [
        f"{experiment['key'][:1]} recon",
        f"{experiment['key'][:1]} heatmap",
        f"{experiment['key'][:1]} pred",
    ]

for row_index, manifest_index in enumerate(selected):
    entry = test_manifest[manifest_index]
    fig = plt.figure(figsize=(15.5, 3.4))
    gs = fig.add_gridspec(1, 2 + 3 * len(EXPERIMENTS), wspace=0.08)
    image_hwc = sel_images[row_index].cpu().permute(1, 2, 0).clamp(0, 1).numpy()

    panels = [(image_hwc, None), (RESULTS["A_baseline"]["gt_masks"][manifest_index], "gray")]
    thresholds_used = []
    for experiment in EXPERIMENTS:
        model_key = experiment["model"]
        scoring = experiment["scoring"]
        recon_hwc = RECONS[model_key][row_index].permute(1, 2, 0).clamp(0, 1).numpy()
        scores_map = SCORES[model_key][scoring][row_index]
        pixel_threshold = RESULTS[experiment["key"]]["pixel_threshold"]
        thresholds_used.append(f"{experiment['key'][:1]}={pixel_threshold:.4f}")
        pred_mask = apply_threshold(scores_map.numpy(), pixel_threshold)
        heat_rgb = score_to_heatmap(scores_map, vmin=0.0, vmax=heat_vmax)
        panels += [(recon_hwc, None), (heat_rgb, None), (pred_mask, "gray")]

    for col, (panel, cmap) in enumerate(panels):
        axis = fig.add_subplot(gs[0, col])
        if cmap is not None:
            axis.imshow(panel, cmap=cmap, vmin=0.0, vmax=1.0)
        else:
            axis.imshow(np.asarray(panel))
        axis.set_title(panel_titles[col], fontsize=7)
        axis.axis("off")

    label = "DEFECTIVE" if entry["is_defect"] else "NORMAL"
    plt.suptitle(
        f"{entry['name']}  ({label}, {entry['defect']})   |   heatmap 0..{heat_vmax:.3f} pinned   |   "
        f"pred thresholds from normal val: {', '.join(thresholds_used)}",
        fontsize=9,
    )
    out_path = FIGURES_DIR / f"compare_{entry['defect']}_{Path(entry['name']).stem}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved: {out_path}")

print(f"comparison figures saved under {FIGURES_DIR}")
VIZ_OK = True

## Section 11 - Save everything needed to audit or redo this comparison

`outputs/runs/phase4_comparison/comparison.json` records: the research
question, the four configurations and their checkpoint paths/ids, every
fixed-a-priori constant (λ, scales, weights, percentiles, seed), training
summaries, common-currency validation MSE, all metrics and thresholds, the
hypothesis verdicts computed above, the environment, and the limitations
below. Per-run directories keep their Phase-2-style artifacts
(`checkpoints/`, `history.json`, `config.yaml`, `metadata.json`).

**Limitations recorded with the results (read before citing any number):**

1. one random seed - no error bars; seed sensitivity unmeasured;
2. one MVTec category (`bottle`) - no claim about other categories;
3. one training pair per loss - checkpoint variance unmeasured;
4. λ, scales, weights and percentiles fixed a priori - not swept, so this is
   not an exhaustive optimisation of either improvement;
5. both models are small/from-scratch baselines - conclusions may not
   transfer to larger architectures or pretrained features;
6. threshold-based metrics (Dice/IoU) use the fixed P99.5 protocol; other
   operating points would move them (AUROC/AP would not move);
7. the harness/smoke overrides (`NB05_EPOCHS`, `NB05_IMAGE_SIZE`) produce
   *reduced* runs whose numbers are not comparable to the full experiment.

In [ ]:
# Persist comparison.json + per-configuration metric summaries.
import platform
import time as _time

comparison = {
    "notebook": "05_research_improvements.ipynb",
    "created": _time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    "research_question": (
        "Do multi-scale scoring and/or an edge-aware reconstruction loss improve "
        "reconstruction-based anomaly detection over the Phase-2/3 baseline?"
    ),
    "seed": cfg.seed,
    "category": cfg.category,
    "fixed_a_priori": {
        "edge_weight_lambda": EDGE_WEIGHT,
        "multiscale_scales": list(MS_SCALES),
        "multiscale_weights": list(MS_WEIGHTS),
        "pixel_percentile": PIXEL_PERCENTILE,
        "image_percentile": IMAGE_PERCENTILE,
        "threshold_source": "held-out normal train/good validation split only",
        "optimizer": cfg.train.optimizer,
        "lr": cfg.train.lr,
        "epochs": cfg.train.epochs,
        "batch_size": cfg.data.batch_size,
        "image_size": cfg.data.image_size,
        "model": {
            "base_channels": cfg.model.base_channels,
            "channel_mults": list(cfg.model.channel_mults),
            "latent_channels": cfg.model.latent_channels,
            "num_convs_per_stage": cfg.model.num_convs_per_stage,
        },
    },
    "training": {
        "baseline": {
            "run_dir": str(BASELINE_RUN_DIR),
            "checkpoint": str(BASELINE_CKPT),
            "checkpoint_id": CHECKPOINT_IDS["baseline"],
            "criterion": "MSE (ReconstructionLoss)",
            "best_recorded_val_loss": baseline_result.best_metric,
            "plain_val_mse": val_mse_baseline,
            "epochs": len(baseline_result.records),
            "seconds": baseline_result.total_seconds,
        },
        "edge_aware": {
            "run_dir": str(EDGE_RUN_DIR),
            "checkpoint": str(EDGE_CKPT),
            "checkpoint_id": CHECKPOINT_IDS["edge"],
            "criterion": f"MSE + {EDGE_WEIGHT} * gradient_magnitude_mse (EdgeAwareReconstructionLoss)",
            "best_recorded_val_loss": edge_result.best_metric,
            "plain_val_mse": val_mse_edge,
            "epochs": len(edge_result.records),
            "seconds": edge_result.total_seconds,
        },
    },
    "results": {
        experiment["key"]: {
            "title": experiment["title"],
            "model_run": experiment["model"],
            "scoring": experiment["scoring"],
            "pixel_threshold": RESULTS[experiment["key"]]["pixel_threshold"],
            "image_threshold": RESULTS[experiment["key"]]["image_threshold"],
            "threshold_percentiles": {
                "pixel": RESULTS[experiment["key"]]["pixel_percentile"],
                "image": RESULTS[experiment["key"]]["image_percentile"],
            },
            "n_val_threshold_source": RESULTS[experiment["key"]]["n_val_source"],
            "metrics": RESULTS[experiment["key"]]["metrics"],
        }
        for experiment in EXPERIMENTS
    },
    "hypothesis_verdicts": {
        "H1_multiscale_pixel_auroc_delta": h1_pixel,
        "H1_multiscale_image_auroc_delta": h1_image,
        "H2_edge_pixel_auroc_delta": h2_pixel,
        "H2_edge_dice_delta": h2_dice,
        "H3_combined_vs_best_single_pixel_auroc_delta": h3_pixel,
        "note": "verdicts derive from this single-seed run only",
    },
    "figures": sorted(p.name for p in FIGURES_DIR.glob("*.png")),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "device": str(DEVICE),
        "smoke_overrides": {k: v for k, v in OVERRIDES.items()} or None,
    },
    "limitations": [
        "single random seed - no error bars",
        "single MVTec category (bottle)",
        "one training pair per loss variant",
        "lambda / scales / weights / percentiles fixed a priori, not swept",
        "small from-scratch architecture - conclusions may not transfer",
        "Dice/IoU tied to the P99.5 threshold protocol; AUROC/AP are threshold-free",
        "smoke-run numbers (NB05_EPOCHS/NB05_IMAGE_SIZE) are not full-experiment results",
    ],
}

comparison_path = PHASE4_DIR / "comparison.json"
comparison_path.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(f"saved: {comparison_path}  ({comparison_path.stat().st_size / 1024:.1f} KB)")

gitignore_text = (REPO_ROOT / ".gitignore").read_text(encoding="utf-8")
artifacts_gitignored = any(
    line.strip() in ("/outputs/", "outputs/") for line in gitignore_text.splitlines()
)
assert artifacts_gitignored, ".gitignore no longer excludes outputs/ - artifacts could leak into git"
print(f".gitignore covers outputs/: {artifacts_gitignored}")
SAVE_OK = True

## Final checklist

Every entry must pass in *this* run. The checklist is computed from the flags
set by the cells above - nothing is assumed.

In [ ]:
# Final checklist - all sections of notebook 05.
final_checks = {
    "Section 1  - src modules imported + hashed (Phase-4 files present)": SRC_OK,
    "Section 2  - dataset scanned; train/val split audited (train/good only)": SCAN_OK and LOADERS_OK and CFG_OK,
    "Section 3  - multi-scale scoring verified (baseline equivalence + hand example)": MULTISCALE_OK,
    "Section 4  - edge-aware loss verified (algebra + gradients + error paths)": EDGE_LOSS_OK,
    "Section 5  - experiment matrix + fixed constants declared": MATRIX_OK,
    "Section 6  - baseline trained from random init (MSE)": BASELINE_TRAINED,
    "Section 7  - edge-aware trained from same init (MSE + lambda grad)": EDGE_TRAINED and COMMON_VAL_MSE_OK,
    "Section 8  - all 4 configurations evaluated with one protocol": MANIFEST_OK and EVAL_OK,
    "Section 9  - comparison table + hypothesis verdicts computed": TABLE_OK and BARS_OK,
    "Section 10 - per-image comparison figures rendered": VIZ_OK,
    "Section 11 - comparison.json saved, limitations recorded, outputs gitignored": SAVE_OK,
}

width = max(len(name) for name in final_checks)
all_ok = True
for name, ok in final_checks.items():
    ok = bool(ok)
    all_ok = all_ok and ok
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
assert all_ok, "Phase 4 checklist failed - fix the failing section before reporting results."

print(f"\nbaseline checkpoint : {BASELINE_CKPT}  (id {CHECKPOINT_IDS['baseline']})")
print(f"edge checkpoint     : {EDGE_CKPT}  (id {CHECKPOINT_IDS['edge']})")
print(f"lambda              : {EDGE_WEIGHT}   scales {MS_SCALES}  weights {MS_WEIGHTS}")
print(f"thresholds          : pixel P{PIXEL_PERCENTILE} / image P{IMAGE_PERCENTILE} of normal val scores")
print(f"\nheadline metrics (this run):")
for experiment in EXPERIMENTS:
    m = RESULTS[experiment["key"]]["metrics"]
    print(
        f"  {experiment['key']:<13} imgAUROC {m['image_auroc']:.4f}  pxAUROC {m['pixel_auroc']:.4f}  "
        f"AP {m['pixel_ap']:.4f}  Dice {m['dice_defective_mean']:.4f}  IoU {m['iou_defective_mean']:.4f}"
    )
print(f"\nFull audit trail: {comparison_path}")
print("Report numbers only together with: single seed, single category, fixed-a-priori hyperparameters.")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `FileNotFoundError: ... Phase 4 added src/...` | runtime still holds the Phase-3 upload | re-upload the repository folder, re-run Section 1 |
| `FileNotFoundError: ... MVTEC` | dataset not extracted on this runtime | run notebook 02, or set `MVTEC_ROOT` |
| training much slower than ~3 min/30 epochs | CPU-only runtime | fine for smoke runs; full experiment wants a GPU |
| `NB05_*` numbers look poor | smoke run: fewer epochs / smaller images | expected; only default settings are the experiment |
| comparison.json missing | Section 11 not reached (an earlier cell failed) | fix the failing cell; re-run from there |